# Asynchronous Programming in Python with `asyncio` — `async`/`await`, Event Loops & Concurrent I/O

> **Topic:** Asynchronous Programming (`asyncio`) | **Folder:** Concurrency

**Asynchronous programming**, supported natively in Python via **`asyncio`**, allows code to execute
**without blocking the main thread** using the `async` and `await` syntax.

Unlike multithreading (which relies on OS context switching) or multiprocessing (which spawns heavy separate processes),
`asyncio` uses a single-threaded **Event Loop** to cooperatively schedule thousands of concurrent non-blocking I/O operations
(such as web API calls, database queries, socket networking, or file I/O) with minimal memory overhead.

---

## Table of Contents
1. [Asynchrony Concepts & The Event Loop](#1.-Asynchrony-Concepts-&-The-Event-Loop)
2. [Coroutines, `async def` & `await` Syntax](#2.-Coroutines,-`async-def`-&-`await`-Syntax)
3. [Scheduling Tasks Concurrently (`asyncio.create_task`)](#3.-Scheduling-Tasks-Concurrently-(asyncio.create_task))
4. [Gathering Results (`asyncio.gather` & `asyncio.as_completed`)](#4.-Gathering-Results-(asyncio.gather-&-asyncio.as_completed))
5. [Timeouts, Cancellation & Exception Handling](#5.-Timeouts,-Cancellation-&-Exception-Handling)
6. [Async Context Managers (`async with`) & Async Iterators (`async for`)](#6.-Async-Context-Managers-(async-with)-&-Async-Iterators-(async-for))
7. [Concurrency Control: `asyncio.Semaphore` & `asyncio.Queue`](#7.-Concurrency-Control:-asyncio.Semaphore-&-asyncio.Queue)
8. [Offloading Blocking Sync Code (`asyncio.to_thread`)](#8.-Offloading-Blocking-Sync-Code-(asyncio.to_thread))
9. [Empirical Benchmark: Synchronous vs. Asynchronous I/O](#9.-Empirical-Benchmark:-Synchronous-vs.-Asynchronous-I/O)
10. [Quick Reference Card](#10.-Quick-Reference-Card)


---
## 1. Asynchrony Concepts & The Event Loop

- **Event Loop**: A continuous loop that manages, dispatches, and executes asynchronous tasks.
- **Cooperative Multitasking**: Tasks voluntarily yield control back to the event loop when waiting for I/O (`await`).

| Approach | Model | Memory Overhead | Max Concurrent Connections |
|----------|-------|-----------------|----------------------------|
| **Synchronous** | Sequential blocking | Low | 1 at a time |
| **Multithreading** | Preemptive OS threads | High (~8MB per thread) | ~1,000 threads max |
| **Multiprocessing** | Isolated OS processes | Very High (Full Python copies) | Limited by CPU cores |
| **`asyncio` (Async)** | Single-thread Event Loop | **Very Low (~KB per task)** | **10,000+ connections** |


---
## 2. Coroutines, `async def` & `await` Syntax

- `async def`: Defines a **coroutine function**.
- `await`: Pauses execution until the awaited coroutine/future completes, releasing control to the event loop.


In [ ]:
import asyncio
import time

async def fetch_data(source_id, delay):
    print(f"  [Fetch-{source_id}] Starting request (simulating {delay}s network latency)..." )
    await asyncio.sleep(delay)  # Non-blocking pause!
    print(f"  [Fetch-{source_id}] Data received successfully.")
    return f"Payload_{source_id}"

# In Jupyter, we can run coroutines directly using await!
res = await fetch_data(1, 0.2)
print("Result:", res)


---
## 3. Scheduling Tasks Concurrently (`asyncio.create_task`)

`asyncio.create_task(coro)` schedules a coroutine to run **immediately in the background** on the event loop.


In [ ]:
async def run_concurrent_tasks():
    print("--- Scheduling Tasks Concurrently ---")
    # Schedule tasks on event loop
    task1 = asyncio.create_task(fetch_data("A", 0.3))
    task2 = asyncio.create_task(fetch_data("B", 0.2))
    task3 = asyncio.create_task(fetch_data("C", 0.1))
    
    # Await results (Task C finishes first!)
    res1 = await task1
    res2 = await task2
    res3 = await task3
    return [res1, res2, res3]

results = await run_concurrent_tasks()
print("All Results:", results)


---
## 4. Gathering Results (`asyncio.gather` & `asyncio.as_completed`)

- **`asyncio.gather(*coros)`**: Runs multiple coroutines concurrently and returns their outputs in **original list order**.
- **`asyncio.as_completed(tasks)`**: Yields results **as soon as each individual task completes**.


In [ ]:
async def demo_gather_and_completed():
    tasks = [fetch_data(i, delay) for i, delay in enumerate([0.4, 0.1, 0.3], start=1)]
    
    # 1. asyncio.gather
    print("--- Demonstrating asyncio.gather ---")
    gather_res = await asyncio.gather(*tasks)
    print("Gather Output:", gather_res)
    
    # 2. asyncio.as_completed
    print("\n--- Demonstrating asyncio.as_completed ---")
    new_tasks = [fetch_data(f"Fast-{i}", delay) for i, delay in enumerate([0.3, 0.1], start=1)]
    for completed_coro in asyncio.as_completed(new_tasks):
        result = await completed_coro
        print("Finished early:", result)

await demo_gather_and_completed()


---
## 5. Timeouts, Cancellation & Exception Handling


In [ ]:
async def slow_network_call():
    await asyncio.sleep(5.0)
    return "Data"

async def demo_timeout():
    try:
        # Set 0.3s timeout threshold
        res = await asyncio.wait_for(slow_network_call(), timeout=0.3)
    except asyncio.TimeoutError:
        print("Caught asyncio.TimeoutError: Operation exceeded 0.3s timeout limit!")

await demo_timeout()


---
## 6. Async Context Managers (`async with`) & Async Iterators (`async for`)


In [ ]:
class AsyncDatabaseConnection:
    async def __aenter__(self):
        print("  [Async DB] Connecting to database...")
        await asyncio.sleep(0.1)
        print("  [Async DB] Connected.")
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("  [Async DB] Closing connection...")
        await asyncio.sleep(0.1)
        print("  [Async DB] Connection closed.")

    async def query(self, sql):
        await asyncio.sleep(0.1)
        return [{"id": 1, "name": "Alice"}]

async def demo_async_with():
    async with AsyncDatabaseConnection() as db:
        records = await db.query("SELECT * FROM users")
        print("  Query result:", records)

await demo_async_with()


---
## 7. Concurrency Control: `asyncio.Semaphore` & `asyncio.Queue`

A **`Semaphore`** limits the maximum number of concurrent operations (e.g. rate-limiting web requests).


In [ ]:
async def limited_worker(sem, item_id):
    async with sem:  # Max concurrency constrained by semaphore
        print(f"  [Worker-{item_id}] Acquired slot, processing...")
        await asyncio.sleep(0.2)
        print(f"  [Worker-{item_id}] Released slot.")

async def demo_semaphore():
    sem = asyncio.Semaphore(2)  # Allow max 2 concurrent workers
    tasks = [limited_worker(sem, i) for i in range(1, 5)]
    await asyncio.gather(*tasks)

await demo_semaphore()


---
## 8. Offloading Blocking Sync Code (`asyncio.to_thread`)

Synchronous blocking calls (e.g., standard file I/O, `time.sleep()`, heavy math) freeze the Event Loop.
Use **`asyncio.to_thread(func, *args)`** to run blocking code in a background OS worker thread!


In [ ]:
def blocking_file_operation(filename):
    time.sleep(0.2)  # Synchronous blocking sleep
    return f"Contents of {filename}"

async def demo_to_thread():
    print("Offloading synchronous call to background thread...")
    result = await asyncio.to_thread(blocking_file_operation, "data.txt")
    print("Thread result:", result)

await demo_to_thread()


---
## 9. Empirical Benchmark: Synchronous vs. Asynchronous I/O


In [ ]:
# Benchmarking 10 simulated API requests with 0.1s latency
def sync_fetch(id_):
    time.sleep(0.1)
    return id_

async def async_fetch(id_):
    await asyncio.sleep(0.1)
    return id_

# 1. Synchronous Sequential Execution
t0 = time.perf_counter()
sync_res = [sync_fetch(i) for i in range(10)]
t_sync = time.perf_counter() - t0

# 2. Asynchronous Concurrent Execution
t0 = time.perf_counter()
async_res = await asyncio.gather(*(async_fetch(i) for i in range(10)))
t_async = time.perf_counter() - t0

print(f"10 Requests Synchronous Time : {t_sync:.4f} seconds")
print(f"10 Requests Asynchronous Time: {t_async:.4f} seconds")
print(f"Async Speedup: ~{t_sync / t_async:.1f}x faster!")


---
## 10. Quick Reference Card


In [ ]:
# ==================================================================
# ASYNCHRONOUS PROGRAMMING (asyncio) – QUICK REFERENCE
# ==================================================================
import asyncio

# Define coroutine:
# async def fetch(): await asyncio.sleep(1); return "done"

# Run concurrently:
# results = await asyncio.gather(fetch(), fetch())

# Rate limiting:
# sem = asyncio.Semaphore(5)
# async with sem: await fetch()


---
## Summary

| Technique | API / Construct | Primary Application |
|-----------|-----------------|---------------------|
| **Coroutine Definition** | `async def` / `await` | Declaring non-blocking functions |
| **Task Scheduling** | `asyncio.create_task()` | Background execution on event loop |
| **Concurrent Gathering** | `asyncio.gather(*tasks)` | Aggregating outputs of multiple I/O calls |
| **Timeout Protection** | `asyncio.wait_for(coro, timeout)` | Preventing hung I/O requests |
| **Rate Limiting** | `asyncio.Semaphore(limit)` | Restricting concurrent API/DB calls |
| **Sync Interoperability** | `asyncio.to_thread(func)` | Offloading blocking sync code |

---
*Next up: **Multithreading & Advanced Concurrency Patterns***
